# Train a Custom NER Model (spaCy)

This notebook trains a Named Entity Recognition (NER) model that can identify monetary amounts in transaction text.

Instead of using complex regex patterns to find amounts, we teach a small neural network to recognise them — the same way humans do, by understanding the surrounding context.

### What is NER?
NER (Named Entity Recognition) is a type of "Token Classification". Instead of classifying the WHOLE sentence (like our DistilBERT model does for category), NER classifies EACH WORD individually.

**Example:**
- **Input:**  `Paid 450 rupees for pizza via GPay`
- **Output:** `Paid [450]_AMOUNT rupees for pizza via GPay`

In [1]:
import os
import re
import random
import pandas as pd
import spacy
from spacy.training import Example

DATASET_PATH = "eda_dataset_v3.csv"
MODEL_OUTPUT_DIR = "amount_ner_model"

### Step 1: Generate NER Training Data from CSV

We read the existing dataset CSV and automatically create NER training examples by finding the amount value inside each sentence.

In [2]:
def generate_ner_training_data(csv_path: str) -> list[tuple[str, dict]]:
    df = pd.read_csv(csv_path)

    # Only keep rows that have a valid numeric amount
    df = df.dropna(subset=["text", "amount"])
    df["text"] = df["text"].astype(str).str.strip()
    df["amount"] = df["amount"].astype(str).str.strip()

    training_data = []

    for _, row in df.iterrows():
        text = row["text"]
        amount_str = row["amount"]

        try:
            amount_val = float(amount_str)
        except (ValueError, TypeError):
            continue

        amount_val = abs(amount_val)
        if amount_val <= 0:
            continue

        search_patterns = []

        if amount_val == int(amount_val):
            search_patterns.append(str(int(amount_val)))
        else:
            search_patterns.append(f"{amount_val:.2f}")
            search_patterns.append(str(amount_val))

        found = False
        for pattern in search_patterns:
            match = re.search(r'(?<!\d)' + re.escape(pattern) + r'(?!\d)', text)
            if match:
                start, end = match.span()
                entities = [(start, end, "AMOUNT")]
                training_data.append((text, {"entities": entities}))
                found = True
                break

    print(f"Generated {len(training_data)} NER training examples from {len(df)} rows")
    return training_data

data = generate_ner_training_data(DATASET_PATH)

Generated 42900 NER training examples from 43923 rows


### Step 2: Train the spaCy NER Model

We train a spaCy NER model to recognise `AMOUNT` entities.

In [3]:
def train_ner_model(training_data, output_dir, n_iter=30):
    nlp = spacy.blank("en")
    ner = nlp.add_pipe("ner")
    ner.add_label("AMOUNT")

    random.shuffle(training_data)

    split_idx = int(len(training_data) * 0.9)
    train_set = training_data[:split_idx]
    val_set = training_data[split_idx:]

    print(f"\nTraining: {len(train_set)} examples")
    print(f"Validation: {len(val_set)} examples")
    print(f"Training for {n_iter} iterations...\n")

    train_examples = []
    for text, annotations in train_set:
        doc = nlp.make_doc(text)
        example = Example.from_dict(doc, annotations)
        train_examples.append(example)

    optimizer = nlp.begin_training()

    for iteration in range(n_iter):
        random.shuffle(train_examples)
        losses = {}

        for batch_start in range(0, len(train_examples), 32):
            batch = train_examples[batch_start : batch_start + 32]
            nlp.update(batch, sgd=optimizer, losses=losses)

        # Evaluate on validation set
        correct = 0
        total = 0
        for text, annotations in val_set:
            doc = nlp(text)
            predicted_amounts = {ent.text for ent in doc.ents if ent.label_ == "AMOUNT"}
            true_entities = annotations["entities"]

            for start, end, label in true_entities:
                total += 1
                true_text = text[start:end]
                if true_text in predicted_amounts:
                    correct += 1

        accuracy = correct / total if total > 0 else 0
        print(f"  Iteration {iteration + 1:>3}/{n_iter}  |  Loss: {losses.get('ner', 0):.4f}  |  Val Accuracy: {accuracy:.2%}")

    os.makedirs(output_dir, exist_ok=True)
    nlp.to_disk(output_dir)
    print(f"\n✓ Model saved to: {output_dir}")

train_ner_model(data, MODEL_OUTPUT_DIR, n_iter=30)


Training: 38610 examples
Validation: 4290 examples
Training for 30 iterations...



/usr/local/lib/python3.12/dist-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "H&M order 11704.00 paid by card" with entities "[(10, 15, 'AMOUNT')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Paid DTH recharge for 886.00 using bhim from Union..." with entities "[(22, 25, 'AMOUNT')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "paid 1127.00 to JioCinema via slice" with entities "[(5, 9, 'AMOUNT

  Iteration   1/30  |  Loss: 1656.0558  |  Val Accuracy: 99.70%
  Iteration   2/30  |  Loss: 0.0000  |  Val Accuracy: 99.70%
  Iteration   3/30  |  Loss: 0.0000  |  Val Accuracy: 99.70%
  Iteration   4/30  |  Loss: 0.0000  |  Val Accuracy: 99.70%
  Iteration   5/30  |  Loss: 0.0000  |  Val Accuracy: 99.70%
  Iteration   6/30  |  Loss: 0.0000  |  Val Accuracy: 99.70%
  Iteration   7/30  |  Loss: 0.0000  |  Val Accuracy: 99.70%
  Iteration   8/30  |  Loss: 0.0000  |  Val Accuracy: 99.70%
  Iteration   9/30  |  Loss: 0.0000  |  Val Accuracy: 99.70%
  Iteration  10/30  |  Loss: 0.0000  |  Val Accuracy: 99.70%
  Iteration  11/30  |  Loss: 0.0000  |  Val Accuracy: 99.70%
  Iteration  12/30  |  Loss: 0.0000  |  Val Accuracy: 99.70%
  Iteration  13/30  |  Loss: 0.0000  |  Val Accuracy: 99.70%
  Iteration  14/30  |  Loss: 0.0000  |  Val Accuracy: 99.70%
  Iteration  15/30  |  Loss: 0.0000  |  Val Accuracy: 99.70%
  Iteration  16/30  |  Loss: 0.0000  |  Val Accuracy: 99.70%
  Iteration  17/30  |

### Step 3: Quick Test

In [4]:
loaded_nlp = spacy.load(MODEL_OUTPUT_DIR)
test_sentences = [
    "Bhai Swiggy se pizza mangwaya 450 rupaye ka",
    "Amazon pe 2000 ka shopping kiya hdfc card se",
    "Ola cab liya 350 rupaye cash diye",
    "Netflix subscription 199 rupaye renew kiya",
    "Electricity bill 1200 rupaye card se pay kiya",
]

for sent in test_sentences:
    doc = loaded_nlp(sent)
    amounts = [(ent.text, ent.label_) for ent in doc.ents]
    print(f"  Input:  {sent}")
    print(f"  Found:  {amounts}\n")

  Input:  Bhai Swiggy se pizza mangwaya 450 rupaye ka
  Found:  [('450', 'AMOUNT')]

  Input:  Amazon pe 2000 ka shopping kiya hdfc card se
  Found:  [('2000', 'AMOUNT')]

  Input:  Ola cab liya 350 rupaye cash diye
  Found:  [('350', 'AMOUNT')]

  Input:  Netflix subscription 199 rupaye renew kiya
  Found:  [('199', 'AMOUNT')]

  Input:  Electricity bill 1200 rupaye card se pay kiya
  Found:  [('1200', 'AMOUNT')]



In [5]:
import shutil
import os

print(f"Zipping {MODEL_OUTPUT_DIR} for easy download...")
shutil.make_archive(MODEL_OUTPUT_DIR, 'zip', MODEL_OUTPUT_DIR)
print(f"Created {MODEL_OUTPUT_DIR}.zip")

Zipping amount_ner_model for easy download...
Created amount_ner_model.zip
